# กด Runtime > Run all

In [1]:
# @title
import pandas as pd
import os
import numpy as np


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# @title
BASE_DIR = os.getcwd()
speaker_dict_dir = os.path.join(BASE_DIR, 'SpeakerTypeDict')
rawdata_dir = os.path.join(BASE_DIR, 'Input')
output_dir = os.path.join(BASE_DIR, 'Output')


In [3]:
# @title
import re
from datetime import datetime

excel_files = [f for f in os.listdir(speaker_dict_dir) if f.endswith((".xlsx", ".xls"))]

if excel_files:
    # Sort to ensure consistent first file if needed, or just take the first from os.listdir
    excel_files.sort()
    first_excel_file = excel_files[0]
    file_path = os.path.join(speaker_dict_dir, first_excel_file)
    df_speaker_type = pd.read_excel(file_path)
    print(f"Successfully loaded the first Excel file found: {first_excel_file}")
    print(len(df_speaker_type))
else:
    print(f"No Excel files found in the directory: {speaker_dict_dir}")
    df_speaker_type = pd.DataFrame() # Initialize as empty DataFrame

# Strip whitespace and lowercase usernames for exact matching
df_speaker_type['Username_clean'] = (
    df_speaker_type['Username'].astype(str).str.strip().str.lower()
)

# Build dictionary
speakertype_dict = dict(
    zip(
        df_speaker_type['Username_clean'],
        'Type of Speaker/' + df_speaker_type['Speaker Type'].astype(str),
    )
)



Successfully loaded the first Excel file found: SpeakerTypeDict_20260622.xlsx
1364
{'page': 'Type of Speaker/Type of Speaker/Brand Voice', 'คลินิกทันตกรรมฟ้าใส ทำฟัน จัดฟัน ระยอง จัดฟันแบบใส invisalign rayong': 'Type of Speaker/Type of Speaker/Influencer & Page', 'หุ้นสมาร์ท - hoonsmart : เสนอความจริง ทุกการลงทุน': 'Type of Speaker/Type of Speaker/Publisher', 'หนังสือพิมพ์รวมพลัง': 'Type of Speaker/Type of Speaker/Publisher', 'tnn': 'Type of Speaker/Type of Speaker/Publisher', '@mod_x': 'Type of Speaker/Type of Speaker/Influencer & Page', 'palm studio official': 'Type of Speaker/Type of Speaker/Influencer & Page', 'or official': 'Type of Speaker/Type of Speaker/Brand Voice', 'น้องปอสาม': 'Type of Speaker/Type of Speaker/Influencer & Page', 'asie-asia': 'Type of Speaker/Type of Speaker/Influencer & Page', 'เกาะกระแสเศรษฐกิจ': 'Type of Speaker/Type of Speaker/Publisher', 'สำนักข่าวอิศรา': 'Type of Speaker/Type of Speaker/Publisher', 'ac news': 'Type of Speaker/Type of Speaker/Publisher',

In [4]:
# @title
excel_files = [f for f in os.listdir(rawdata_dir) if f.endswith(('.xlsx', '.xls'))]

if excel_files:
    # Sort to ensure consistent 'first' file if needed, or just take the first from os.listdir
    excel_files.sort()
    first_excel_file = excel_files[0]
    file_path = os.path.join(rawdata_dir, first_excel_file)
    rawdata_df = pd.read_excel(file_path)
    print(f"Successfully loaded the first Excel file found: {first_excel_file}")
    print(len(rawdata_df))
else:
    print(f"No Excel files found in the directory: {rawdata_dir}")
    rawdata_df = pd.DataFrame() # Initialize as empty DataFrame

    # change Content column name to Sample Content
    rawdata_df.rename(columns={'Content': 'Sample Content'}, inplace=True)

Successfully loaded the first Excel file found: SCG_TalkWalkerRaw_1-26Jul.xlsx
30737
                                                     url  \
30732  http://twitter.com/TY_TT0701/status/2073359600...   
30733  https://www.jobpub.com/%E0%B8%AB%E0%B8%B2%E0%B...   
30734  https://www.facebook.com/624866954332213_14662...   
30735  http://twitter.com/ChandlerRu73369/status/2073...   
30736  https://market.xn--22c9cn4a4b1f.com/index.php/...   

                    published  \
30732 2026-07-04 17:53:44.000   
30733 2026-07-01 14:03:21.903   
30734 2026-07-07 15:05:52.000   
30735 2026-07-03 21:32:11.000   
30736 2026-07-21 01:44:49.000   

                                                   title  \
30732                                                NaN   
30733  หา​งาน บุญ​ถาวร เปิด​รับ​สมัคร​งาน​ด่วน ตำแหน่...   
30734                                                NaN   
30735                                                NaN   
30736  ผู้​ผลิต​น้ำมัน​ยู​คา​ลิ​ปตัส น้ำมัน​ตะไคร้หอม..

In [5]:
# @title
import re

def extract_domain_name(url):
    """
    Extracts the main domain name from a URL, removing 'www.' prefix
    and top-level domain suffixes (like .com, .org, etc.).
    Example: 'www.prachachart.com' becomes 'prachachart'
    """
    if pd.isna(url) or not isinstance(url, str):
        return None

    # Remove 'http://', 'https://'
    url = re.sub(r'https?://', '', url)
    # Remove 'www.' prefix
    url = re.sub(r'^www\.', '', url)

    # Split by '/' to get the domain part if there's a path
    url = url.split('/')[0]

    # Split by '.' and get the second to last part (e.g., 'prachachart' from 'prachachart.com')
    parts = url.split('.')
    if len(parts) >= 2:
        # Handle cases like example.co.uk by taking the first part if more than 2 parts are left after removing TLD
        if len(parts) > 2 and len(parts[-1]) <=3 and len(parts[-2]) <=3: # simple heuristic for co.uk, com.au etc.
            return parts[-3]
        else:
            return parts[-2]
    elif len(parts) == 1:
        return parts[0]
    return None

def apply_source_type_url_replacement(df):
    """
    Replaces values in 'extra_author_attributes.name' with cleaned URLs
    from the 'URL' column, but only for specific 'source_type' values.
    Returns the modified DataFrame and a boolean mask indicating modified rows.
    """
    df_copy = df.copy() # Work on a copy to avoid modifying the original DataFrame unexpectedly

    allowed_source_types = [
      "BLOG,BLOG_OTHER",
      "ONLINENEWS,ONLINENEWS_OTHER",
      "ONLINENEWS,ONLINENEWS_NEWSPAPER",
      "MESSAGEBOARD,MESSAGEBOARD_OTHER",
      "NEWSLETTER,NEWSLETTER_SUBSTACK",
      "ONLINENEWS,ONLINENEWS_PRESSRELEASES",
      "ONLINENEWS,ONLINENEWS_MAGAZINE",
      "ONLINENEWS,ONLINENEWS_TVRADIO",
      "ONLINENEWS,ONLINENEWS_AGENCY",
      "OTHER"
    ]

    # Create a boolean mask for rows where 'source_type' is in the allowed list
    mask = df_copy['source_type'].isin(allowed_source_types)

    # Apply the URL extraction only to the filtered rows
    df_copy.loc[mask, 'extra_author_attributes.name'] = df_copy.loc[mask, 'url'].apply(extract_domain_name)

    return df_copy, mask

# Assuming rawdata_df exists from previous cells and has 'url', 'source_type', and 'extra_author_attributes.name' columns
rawdata_df, url_replaced_mask = apply_source_type_url_replacement(rawdata_df)

# Add a column to indicate if the URL was replaced in 'extra_author_attributes.name'
rawdata_df['is_url_replaced'] = url_replaced_mask




DataFrame head with 'is_url_replaced' column:


,url,source_type,extra_author_attributes.name,is_url_replaced
0,http://www.facebook.com/100445640022349_103714...,"SOCIALMEDIA,SOCIALMEDIA_FACEBOOK",NaN,False
1,https://www.facebook.com/521410597958096_14436...,"SOCIALMEDIA,SOCIALMEDIA_FACEBOOK",NaN,False
2,https://www.facebook.com/285116394953032_14757...,"SOCIALMEDIA,SOCIALMEDIA_FACEBOOK",SCG Brand,False
3,http://www.facebook.com/100445640022349_103133...,"SOCIALMEDIA,SOCIALMEDIA_FACEBOOK",NaN,False
4,http://www.facebook.com/101268018065199_132594...,"SOCIALMEDIA,SOCIALMEDIA_FACEBOOK",OR Official,False



Sample of rows where 'extra_author_attributes.name' was replaced by URL:


,url,source_type,extra_author_attributes.name,is_url_replaced
14271,http://www.jiaodianzw.cn/caishang/15889.html,"ONLINENEWS,ONLINENEWS_OTHER",jiaodianzw,True
14273,https://hoonsmart.com/archives/427418,"BLOG,BLOG_OTHER",hoonsmart,True
14274,https://www.thestorythailand.com/scgp-q2-2569/,"BLOG,BLOG_OTHER",thestorythailand,True
14279,https://www.prachachat.net/real-estate/news-20...,"ONLINENEWS,ONLINENEWS_NEWSPAPER",prachachat,True
14281,https://www.todayupdatenews.com/2026/07/4-reta...,"BLOG,BLOG_OTHER",todayupdatenews,True


In [6]:
import pandas as pd
import numpy as np

def update_speaker_tags(df, speakertype_dict):
    """Transforms the DataFrame by updating the 'tags_customer' column with
    speaker types across three sequential fallback steps without duplicating existing tags.
    """
    # Create a copy to avoid SettingWithCopyWarning on the original DataFrame
    df = df.copy()

    # Temporarily fill NaN values with empty strings for safe string operations
    df["tags_customer"] = df["tags_customer"].fillna("").astype(str)
    df["tags_internal"] = df["tags_internal"].fillna("").astype(str)

    # Clean dictionary keys (lowercase and strip whitespace) for exact matching
    lowercase_speakertype_dict = {
        str(k).strip().lower(): v for k, v in speakertype_dict.items()
    }

    # =====================================================================
    # STEP 1: Tag remaining rows matching usernames in speakertype_dict
    # =====================================================================
    # Evaluate mask to only look at rows that STILL lack a speaker type
    mask_no_speaker = ~df["tags_customer"].str.contains("Type of Speaker", na=False)

    # Clean author names in the dataframe to match dictionary keys
    author_clean = df["extra_author_attributes.name"].astype(str).str.strip().str.lower()

    # Map the cleaned author names
    mapped_tags = author_clean.map(lowercase_speakertype_dict)

    step1_rows = mask_no_speaker & mapped_tags.notna()
    tags_to_add = mapped_tags[step1_rows]

    df.loc[step1_rows, "tags_customer"] = np.where(
        df.loc[step1_rows, "tags_customer"] == "",
        tags_to_add,
        df.loc[step1_rows, "tags_customer"] + "," + tags_to_add,
    )

    # =====================================================================
    # STEP 2: Tag 'isComment' rows as 'Consumer Voice'
    # =====================================================================
    # Re-evaluate mask because Step 1 just added some speaker tags
    mask_no_speaker = ~df["tags_customer"].str.contains("Type of Speaker", na=False)

    # Adding case=False just in case 'IsComment' varies in capitalization
    is_comment_mask = df["tags_internal"].str.contains("isComment", case=False, na=False)

    step2_rows = mask_no_speaker & is_comment_mask
    tag_consumer = "Type of Speaker/Consumer Voice"

    df.loc[step2_rows, "tags_customer"] = np.where(
        df.loc[step2_rows, "tags_customer"] == "",
        tag_consumer,
        df.loc[step2_rows, "tags_customer"] + "," + tag_consumer,
    )

    # =====================================================================
    # STEP 3: Fallback — Tag all leftover rows as 'Influencer & Page'
    # =====================================================================
    # Re-evaluate mask one last time for any rows that slipped through Steps 1 & 2
    mask_leftover = ~df["tags_customer"].str.contains("Type of Speaker", na=False)
    tag_influencer = "Type of Speaker/Influencer & Page"

    df.loc[mask_leftover, "tags_customer"] = np.where(
        df.loc[mask_leftover, "tags_customer"] == "",
        tag_influencer,
        df.loc[mask_leftover, "tags_customer"] + "," + tag_influencer,
    )

    return df

In [7]:
# @title
updated_df = update_speaker_tags(rawdata_df, speakertype_dict)

In [8]:
# @title
output_file_path = os.path.join(output_dir, first_excel_file)

updated_df.to_excel(output_file_path, index=False, engine_kwargs={'options': {'strings_to_urls': False}})
print(f"DataFrame updated_df saved to: {output_file_path}")


DataFrame updated_df saved to: /content/drive/Shareddrives/SCG/SpeakerTypeDict/Output/SCG_TalkWalkerRaw_1-26Jul.xlsx
